---

## **[실습 문제]**

현재 RAG 평가 시스템 흐름에 맞춰, 아래 요구사항을 충족하는 종합 평가 시스템을 구현하세요.
(현재 데이터셋의 `input`은 `{"user_input": "질문", "reference_contexts": "참조 컨텍스트"}` 형태의 딕셔너리입니다.)

### 예시 

1. **Custom Evaluator 추가 (Item-level)**
   - **근거성(Groundedness) LLM 평가자**:
     - OpenEvals의 `create_llm_as_judge`를 활용해 커스텀 LLM 판사를 생성합니다.
     - 생성 답변(`output`)이 `input` 딕셔너리 내의 `reference_contexts`에 잘 근거하고 있는지 0~1 점수로 평가합니다. (질문은 `input["user_input"]` 사용)
   - **키워드 일치 평가자**:
     - 정답(`expected_output`)에 포함된 핵심 키워드(공백 기준 분리)들이 생성된 답변(`output`)에 포함되어 있는지 비율(0~1)을 계산해 반환합니다.

2. **Run-level Evaluator 추가 (Run-level)**
   - **종합 성능 요약 평가자**:
     - 전체 실험 결과(`item_results`)를 받아 ROUGE-L, Groundedness, 키워드 일치 등 모든 평가 지표의 평균 점수를 집계하고 요약 코멘트를 작성합니다.

3. **실험 실행**
   - 정의한 평가자들을 사용하여 실험을 실행하고 결과를 출력합니다.


`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from pprint import pprint
from typing import List, Tuple

`(3) langfuse handler 설정`

In [3]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

https://huggingface.co/datasets/allganize/RAG-Evaluation-Dataset-KO

In [4]:
# 1. 필수 라이브러리 임포트
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. 데이터 로드 (저장해두신 CSV 파일)
df = pd.read_csv("data/rag_test_data.csv")

# ⭐ [추가] 도메인이 '파이낸스'인 데이터만 필터링
# 만약 CSV 파일에 '파이낸스'가 아닌 'Finance' 등 영문으로 적혀있다면 'Finance'로 변경해주세요.
df = df[df['domain'] == 'finance']

# 3. 청크 분할기 설정 (500자 단위, 50자 중첩)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50
)

# 4. 데이터셋을 청크로 변환 및 Document 객체 생성
documents_with_metadata = []

print(f"파이낸스 도메인 데이터 처리 시작... (대상 row 수: {len(df)}개)")

for idx, row in df.iterrows():

    content_with_qa = f"{row['target_answer']}"
    
    # 텍스트를 청크로 분할
    chunks = text_splitter.split_text(content_with_qa)
    
    for chunk in chunks:
        # Document 객체 생성 시 question과 domain을 모두 메타데이터로 관리
        doc = Document(
            page_content=chunk,
            metadata={
                "domain": row.get('domain', 'N/A'),
                "question": row.get('qeustion', 'N/A'), # 오타(qeustion)는 기존 CSV 컬럼명에 맞춰 유지
                "file_name": row.get('target_file_name', 'N/A'),
                "page": row.get('target_page_no', 0)
            }
        )
        documents_with_metadata.append(doc)

# 5. 결과 확인
print(f"완료! 총 {len(documents_with_metadata)}개의 파이낸스 청크가 생성되었습니다.")

# 데이터가 존재할 경우에만 첫 번째 청크와 메타데이터 출력
if documents_with_metadata:
    print("\n[첫 번째 청크 확인]")
    print(f"내용: {documents_with_metadata[0].page_content}")
    print(f"메타데이터: {documents_with_metadata[0].metadata}")
else:
    print("\n조건에 맞는 데이터가 없어 청크가 생성되지 않았습니다. CSV의 'domain' 컬럼 값을 확인해보세요.")

파이낸스 도메인 데이터 처리 시작... (대상 row 수: 60개)
완료! 총 65개의 파이낸스 청크가 생성되었습니다.

[첫 번째 청크 확인]
내용: 시중은행, 지방은행, 인터넷은행 모두 은행업을 영위하기 위해서는 '은행법' 제8조에 근거해 금융위원회의 인가를 필요로 합니다. 그러나 시중은행, 지방은행, 인터넷은행의 인가 요건 및 절차에는 일부 차이가 존재합니다.

첫째, 최저자본금의 경우, 시중은행은 1,000억원이 필요하며, 지방은행과 인터넷은행은 250억원이 필요합니다. 

둘째, 비금융주력자 주식 보유한도의 차이가 있습니다. 시중은행의 경우 4%이며, 지방은행은 15%, 인터넷은행은 34%입니다.

셋째, 영업구역 및 영업방식도 차이가 존재합니다. 시중은행과 지방은행은 온라인과 오프라인을 모두 활용하며 전국 또는 일부 지역에서 영업할 수 있지만, 인터넷전문은행은 전국에서 영업하지만 오직 온라인으로만 이루어집니다.
메타데이터: {'domain': 'finance', 'question': 'N/A', 'file_name': '[별첨] 지방은행의 시중은행 전환시 인가방식 및 절차.pdf', 'page': '4'}


In [5]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# 2. Chroma 벡터스토어 빌드 및 저장 
# 일반 문서용 벡터 DB (비교군)
normal_db = Chroma.from_documents(
    documents=documents_with_metadata,  # Replace with your document list
    embedding=embeddings,    
    collection_name="normal", 
    persist_directory="./chroma_db",
  
)


대답생성

In [6]:
import os
from openai import OpenAI
from langchain_core.documents import Document

# 0. OpenAI 클라이언트 초기화
openai_client = OpenAI()

def generate_reference_answer(question: str, context: str) -> str:
    """질문과 문맥을 기반으로 LLM을 사용해 모범 정답을 생성합니다."""
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system", 
                    "content": "당신은 금융(Finance) 분야 지식 전반에 능통한 전문가입니다. 제공된 [참조 문맥]만을 바탕으로 [질문]에 대한 정확하고 신뢰할 수 있는 모범 답안(Reference Answer)을 한글로 작성해 주세요. 문맥에 없는 내용은 유추하지 마세요. 간결하게 요약하여 1문장으로 간략하게 답변하세요"
                },
                {
                    "role": "user", 
                    "content": f"[질문]: {question}\n\n[참조 문맥]: {context}\n\n"
                }
            ],
            temperature=0.2,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ 정답 생성 중 에러 발생: {e}")
        return "정답 생성 실패 (API 에러)"

# 1. Chroma DB에서 전체 데이터 가져오기
chroma_data = normal_db.get(where={"domain": "finance"}, include=["documents", "metadatas"])

# 2. 가져온 데이터를 기반으로 평가용 데이터셋 변환 + LLM 정답 생성 (최대 100개 제한)
data = []
total_items = len(chroma_data["ids"])

# ⭐ 전체 개수와 100개 중 더 작은 수만큼만 실행합니다.
target_count = min(30, total_items)
print(f"🔄 총 {total_items}개의 데이터 중 최대 {target_count}개에 대해 정답 생성을 시작합니다...")

for i in range(target_count):
    metadata = chroma_data["metadatas"][i] if chroma_data["metadatas"] else {}
    
    # 💡 이전 단계 오타 확인: 만약 질문이 안 뽑히면 "qeustion"으로 키값을 바꿔보세요.
    question = metadata.get("question", metadata.get("qeustion", "질문 없음"))
    context_text = chroma_data["documents"][i]
    context_list = [context_text]
    
    if question == "질문 없음" or not context_text.strip():
        reference_answer = "참조 데이터 부족으로 생성 불가"
    else:
        print(f" [{i+1}/{target_count}] '{question[:15]}...' 정답 생성 중...")
        # LLM 호출하여 정답 자동 생성
        reference_answer = generate_reference_answer(question, context_text)
    
    data.append({
        "user_input": question, 
        "reference": reference_answer,
        "reference_contexts": context_list
    })

print(f"\n✅ 평가용 데이터셋 변환 및 정답(Reference) 생성 완료! 생성된 아이템 수: {len(data)}개")

# 3. 데이터 매핑 검증 출력
if data:
    print("\n" + "="*20 + " 첫 번째 데이터셋 구조 샘플 " + "="*20)
    print(f"👉 질문 (user_input)         : {data[0]['user_input']}")
    print(f"👉 생성된 정답 (reference)    : {data[0]['reference']}")
    print(f"👉 문맥 (reference_contexts): {data[0]['reference_contexts']}")
    print("="*60)

🔄 총 585개의 데이터 중 최대 30개에 대해 정답 생성을 시작합니다...
 [1/30] '시중은행, 지방은행, 인터넷...' 정답 생성 중...
 [2/30] '은행업을 신청하고자 할 때,...' 정답 생성 중...
 [3/30] '본인가를 받으려는 지방은행이...' 정답 생성 중...
 [4/30] '은행법에 의거 예비인가를 신...' 정답 생성 중...
 [5/30] '2019년 YTD 기준으로 ...' 정답 생성 중...
 [6/30] '바이오주 주가 급락에 따른 ...' 정답 생성 중...
 [7/30] '미국, 중국, 한국에 대한 ...' 정답 생성 중...
 [8/30] '2001~2002년과 200...' 정답 생성 중...
 [9/30] '1995년에 연방 기준 금리...' 정답 생성 중...
 [10/30] '2019년 영업이익과 순이익...' 정답 생성 중...
 [11/30] '투자 매력도 상위 국가들의 ...' 정답 생성 중...
 [12/30] '실질금리란 무엇이며 실질금리...' 정답 생성 중...
 [13/30] '2012년 2월부터 신탁은행...' 정답 생성 중...
 [14/30] '2002년과 2012년에 걸...' 정답 생성 중...
 [15/30] '가입기간에 따라 연금 수급액...' 정답 생성 중...
 [16/30] '가입기간에 따라 연금 수급액...' 정답 생성 중...
 [17/30] '2012년 대비 2025년에...' 정답 생성 중...
 [18/30] '고령자의 의료 확보에 관한 ...' 정답 생성 중...
 [19/30] '2005년에 일본에서 제정된...' 정답 생성 중...
 [20/30] '간병 종사자 등의 인재확보를...' 정답 생성 중...
 [21/30] '1994년부터 2001년까지...' 정답 생성 중...
 [22/30] '1996년, 2001년, 2...' 정답 생성 중...
 [23/30] '1996년, 2001년, 2...' 정답 생성 중...
 [24/30] '정부, 기업, 금융산업이

In [7]:
from langfuse import get_client

# Langfuse 클라이언트 초기화
langfuse_client = get_client()
# Langfuse에서 데이터셋 생성
name = "RAG_Evaluation_practice_new_1"

try:
    dataset = langfuse_client.create_dataset(name=name)
    print(f"새 데이터셋 생성: {dataset.name}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"기존 데이터셋 사용: {name}")
        dataset = langfuse_client.get_dataset(name=name)
    else:
        raise e

새 데이터셋 생성: RAG_Evaluation_practice_new_1


In [8]:
for chroma_data in data:
    langfuse_client.create_dataset_item(
        dataset_name=name,
        # 💡 input과 expected_output 내부를 딕셔너리 접근 방식([ "키이름" ])으로 변경했습니다.
        input={
            "user_input": chroma_data["user_input"],
            "reference_contexts": chroma_data["reference_contexts"],

        },
        expected_output=chroma_data["reference"]
    )

In [41]:
from rouge_score import rouge_scorer
from ranx_k.tokenizers import KiwiTokenizer
from openevals.llm import create_llm_as_judge
from openevals.prompts import CONCISENESS_PROMPT,RAG_GROUNDEDNESS_PROMPT

# Kiwi 토크나이저 생성 (tokenize()가 이미 문자열 리스트 반환)
kiwi_tokenizer = KiwiTokenizer(use_stopwords=False, pos_filter=[])

# ROUGE 스코어 계산
rouge_scorer_instance = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    tokenizer=kiwi_tokenizer      # tokenize 메소드를 갖는 토크나이저 사용
)

# 간결성 평가자 (OpenEvals 사용)
conciseness_evaluator = create_llm_as_judge(
    prompt=CONCISENESS_PROMPT,
    feedback_key="conciseness",
    model="openai:gpt-4.1-mini",
)

# 근거성(Groundedness) LLM 평가자
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="groundedness",
    continuous=True,
    model="openai:gpt-4.1-mini",
)

In [42]:
import os
from langfuse import get_client, Evaluation
from rouge_score import rouge_scorer


# ROUGE 스코러 객체 생성 (기존 코드에서 빠져있을 수 있으므로 명시)
rouge_scorer_instance = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# 가정: 기존에 정의된 normal_db 및 conciseness_evaluator(OpenEvals)
normal_retriever = normal_db.as_retriever(search_kwargs={"k": 3})

# ==========================================
# 1. Task 함수 정의 (안전하게 문자열 반환하도록 수정)
# ==========================================
def rag_task(*, item, **kwargs):
    """RAG 체인을 실행하여 최종 '문자열' 답변을 생성"""
    # item.input 호환성 처리
    question = item.input.get("user_input", item.input) if isinstance(item.input, dict) else item.input
    
    # ⚠️ 중요: retriever.invoke()는 List[Document]를 반환하므로 문자열로 가공해야 합니다.
    # 만약 LLM을 결합한 RAG 체인이 있다면 대치하세요 (예: rag_chain.invoke(question))
    docs = normal_retriever.invoke(question)
    
    # 디버깅 및 평가를 위해 검색된 문서들의 본문을 하나의 문자열로 결합합니다.
    answer_text = "\n".join([doc.page_content for doc in docs])
    
    return answer_text


# ==========================================
# 2. 아이템 레벨 Evaluator 정의 (예외 처리 강화)
# ==========================================
def rouge_evaluator(*, input, output, expected_output, **kwargs):
    """ROUGE 점수 평가"""
    # 정답 데이터가 없는 경우 방어 코드
    if not expected_output:
        return Evaluation(name="rouge_l", value=0.0, comment="expected_output이 누락되었습니다.")
        
    try:
        rouge_results = rouge_scorer_instance.score(
            str(expected_output), 
            str(output)
        )
        rouge_l_score = rouge_results['rougeL'].fmeasure
        return Evaluation(
            name="rouge_l",
            value=rouge_l_score,
            comment=f"ROUGE-L F1: {rouge_l_score:.4f}"
        )
    except Exception as e:
        return Evaluation(name="rouge_l", value=0.0, comment=f"에러 발생: {str(e)}")

def conciseness_llm_evaluator(*, input, output, **kwargs):
    """OpenEvals 간결성 평가 (LLM API 호출부)"""
    try:
        user_input = input.get("user_input", input) if isinstance(input, dict) else input
        
        # LLM 호출 시 병목이 생길 수 있으므로 예외 처리를 꼼꼼히 합니다.
        result = conciseness_evaluator(
            inputs=str(user_input),
            outputs=str(output),
        )
        score = 1.0 if result.get('score') else 0.0
        return Evaluation(
            name="conciseness",
            value=score,
            comment=result.get('comment', '')
        )
    except Exception as e:
        # 멈춤 현상을 방지하기 위해 에러 발생 시 대기하지 않고 0.0점 처리 후 넘어가도록 유도
        return Evaluation(name="conciseness", value=0.0, comment=f"LLM 평가 실패: {str(e)}")

def length_evaluator(*, input, output, **kwargs):
    """응답 길이 평가"""
    length = len(str(output))
    return Evaluation(
        name="response_length",
        value=length,
        comment=f"응답 길이: {length}자"
    )


def groundedness_llm_evaluator(*, input, output, expected_output, **kwargs):
    try:
        user_input = input.get("user_input", "") if isinstance(input, dict) else ""
        reference_contexts = input.get("reference_contexts", "") if isinstance(input, dict) else ""
        
        result = groundedness_judge(
            inputs={"question": str(user_input)},
            outputs={"answer": str(output)},
            context={"documents": str(reference_contexts)},
        )
        
        score = result.get('score', 0.0)
        return Evaluation(
            name="groundedness",
            value=float(score) if score is not None else 0.0,
            comment=result.get('comment', '')[:200]
        )
    except Exception as e:
  
        return Evaluation(name="groundedness", value=0.0, comment=str(e))
# 키워드 일치 평가자
def keyword_match_evaluator(*, input, output, expected_output, **kwargs):
    try:
        if not expected_output or not output:
            return Evaluation(name="keyword_match", value=0.0, comment="빈 정답 또는 생성 답변")
            
        keywords = [word.strip() for word in str(expected_output).split() if word.strip()]
        if not keywords:
            return Evaluation(name="keyword_match", value=0.0, comment="정답에 키워드가 없음")
            
        output_str = str(output)
        matched_count = sum(1 for word in keywords if word in output_str)
        score = matched_count / len(keywords)
        
        return Evaluation(
            name="keyword_match",
            value=score,
            comment=f"키워드 매칭 비율: {score:.2f} ({matched_count}/{len(keywords)})"
        )
    except Exception as e:
        return Evaluation(name="keyword_match", value=0.0, comment=str(e))

# ==========================================
# 3. Run-level Evaluator 정의 (전체 실험 집계)
# ==========================================
def average_rouge_evaluator(*, item_results, **kwargs):
    """전체 ROUGE 평균 계산"""
    scores = [
        eval.value for result in item_results
        for eval in result.evaluations
        if eval.name == "rouge_l" and eval.value is not None
    ]
    if not scores:
        return Evaluation(name="avg_rouge_l", value=0.0, comment="계산할 데이터가 없습니다.")
    
    avg = sum(scores) / len(scores)
    return Evaluation(name="avg_rouge_l", value=avg, comment=f"평균 ROUGE-L: {avg:.4f} ({len(scores)}개 항목)")

def average_conciseness_evaluator(*, item_results, **kwargs):
    """전체 간결성 평균 계산"""
    scores = [
        eval.value for result in item_results
        for eval in result.evaluations
        if eval.name == "conciseness" and eval.value is not None
    ]
    if not scores:
        return Evaluation(name="avg_conciseness", value=0.0, comment="계산할 데이터가 없습니다.")
    
    avg = sum(scores) / len(scores)
    return Evaluation(name="avg_conciseness", value=avg, comment=f"평균 간결성: {avg:.2f} ({len(scores)}개 항목)")



In [43]:
# 1. 원래 데이터셋 가져오기
original_dataset = langfuse_client.get_dataset("RAG_Evaluation_practice_new_1")

# 2. 3개만 슬라이싱
original_dataset.items = original_dataset.items[:3]

# 3. 실험 실행
result = original_dataset.run_experiment(
    name="RAG_Evaluation_practice_ROUGE_Test",
    description="ROUGE와 간결성을 활용한 RAG 시스템 샘플 테스트",
    task=rag_task,
    evaluators=[
        rouge_evaluator,
        conciseness_llm_evaluator,
        length_evaluator,
        groundedness_llm_evaluator,
        keyword_match_evaluator
    ],
    run_evaluators=[
        average_rouge_evaluator,
        average_conciseness_evaluator
    ],
    max_concurrency=1,
)

# 결과 출력
print(result.format())

# ==========================================
# ⚠️ [필수 추가] 랭퓨즈 서버로 잔여 데이터 강제 전송
# ==========================================
print("⏳ Langfuse 대시보드로 데이터를 전송 중입니다...")
langfuse_client.flush()
print("✅ 전송 완료! 이제 대시보드를 새로고침해 보세요.")

Individual Results: Hidden (3 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: RAG_Evaluation_practice_ROUGE_Test
📋 Run name: RAG_Evaluation_practice_ROUGE_Test - 2026-07-04T17:34:19.206573Z - ROUGE와 간결성을 활용한 RAG 시스템 샘플 테스트
3 items
Evaluations:
  • keyword_match
  • conciseness
  • groundedness
  • response_length
  • rouge_l

Average Scores:
  • keyword_match: 0.577
  • conciseness: 0.000
  • groundedness: 1.000
  • response_length: 752.000
  • rouge_l: 0.167

Run Evaluations:
  • avg_rouge_l: 0.167
    💭 평균 ROUGE-L: 0.1667 (3개 항목)
  • avg_conciseness: 0.000
    💭 평균 간결성: 0.00 (3개 항목)

🔗 Dataset Run:
   https://jp.cloud.langfuse.com/project/cmqjg28bt005vad0dbvl3wslu/datasets/cmr67lalf02axad0ciuflpxjr/runs/ba1a400a-c6c7-42c2-95a0-bf584530dc2e
⏳ Langfuse 대시보드로 데이터를 전송 중입니다...
✅ 전송 완료! 이제 대시보드를 새로고침해 보세요.


In [ ]:
# LLM-as-Judge Evaluator 구현 (OpenEvals 활용)

from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT, CONCISENESS_PROMPT
from langfuse import Evaluation

# 1. OpenEvals 평가자 생성
correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="correctness",
    model="openai:gpt-4.1-mini",
)

# 2. Langfuse Evaluator로 래핑
def correctness_llm_evaluator(*, input, output, expected_output, **kwargs):
    """LLM-as-Judge 정확성 평가"""
    try:
        user_input = input.get("user_input", input) if isinstance(input, dict) else input
        result = correctness_judge(
            inputs=str(user_input),
            outputs=str(output),
            reference_outputs=str(expected_output),
        )
        
        # score가 boolean인 경우 float로 변환
        score = result.get('score', 0)
        if isinstance(score, bool):
            score = 1.0 if score else 0.0
        
        return Evaluation(
            name="llm_correctness",
            value=float(score),
            comment=result.get('comment', '')[:200]  # 코멘트 길이 제한
        )
    except Exception as e:
        return Evaluation(name="llm_correctness", value=0.0, comment=str(e))


# 3. 커스텀 LLM-as-Judge (한국어)
KOREAN_QUALITY_PROMPT = """당신은 RAG 시스템의 답변 품질을 평가하는 전문가입니다.

<질문>
{inputs}
</질문>

<생성된 답변>
{outputs}
</생성된 답변>

<참조 답변>
{reference_outputs}
</참조 답변>

다음 기준으로 0~1 점수를 부여하세요:
- 1.0: 정확하고 완전한 답변
- 0.7: 대부분 정확하지만 일부 누락
- 0.5: 부분적으로 정확
- 0.3: 관련은 있으나 대부분 부정확
- 0.0: 완전히 잘못됨
"""

korean_quality_judge = create_llm_as_judge(
    prompt=KOREAN_QUALITY_PROMPT,
    continuous=True,  # 0~1 연속 점수
    feedback_key="korean_quality",
    model="openai:gpt-4.1-mini",
)


def korean_quality_evaluator(*, input, output, expected_output, **kwargs):
    """한국어 품질 평가 (연속 점수)"""
    try:
        user_input = input.get("user_input", input) if isinstance(input, dict) else input
        result = korean_quality_judge(
            inputs=str(user_input),
            outputs=str(output),
            reference_outputs=str(expected_output),
        )
        return Evaluation(
            name="korean_quality",
            value=float(result.get('score', 0)),
            comment=result.get('comment', '')[:200]
        )
    except Exception as e:
        return Evaluation(name="korean_quality", value=0.0, comment=str(e))


print("LLM-as-Judge Evaluator 정의 완료")
print("- correctness_llm_evaluator: 정확성 평가 (이진)")
print("- korean_quality_evaluator: 한국어 품질 평가 (연속 점수)")


LLM-as-Judge Evaluator 정의 완료
- correctness_llm_evaluator: 정확성 평가 (이진)
- korean_quality_evaluator: 한국어 품질 평가 (연속 점수)


In [13]:
import asyncio
import time
from langchain_openai import ChatOpenAI
from langfuse import Evaluation

# ============================================
# 비동기 RAG 체인 생성 (ainvoke 지원)
# ============================================

# LangChain의 ainvoke()를 활용한 진짜 비동기 Task
async def async_rag_task(*, item, **kwargs):
    """비동기 RAG 체인 실행 - LangChain ainvoke() 활용"""
    question = item.input if hasattr(item, 'user_input') else item["user_input"]
    if isinstance(question, dict):
        question = question.get("user_input", question)
    # LangChain Runnable은 ainvoke()로 비동기 실행 가능
    answer = await normal_retriever.ainvoke(question)
    return answer

# 비동기 Evaluator 정의
async def async_rouge_evaluator(*, input, output, expected_output, **kwargs):
    """비동기 ROUGE 평가"""
    rouge_results = rouge_scorer_instance.score(
        str(expected_output), 
        str(output)
    )
    score = rouge_results['rougeL'].fmeasure
    return Evaluation(
        name="rouge_l",
        value=score,
        comment=f"ROUGE-L F1: {score:.4f}"
    )

async def async_length_evaluator(*, input, output, **kwargs):
    """비동기 응답 길이 평가"""
    length = len(str(output))
    return Evaluation(
        name="response_length",
        value=length,
        comment=f"응답 길이: {length}자"
    )

print("비동기 함수 정의 완료")
print("- async_rag_task: LangChain ainvoke()를 사용한 비동기 RAG Task")
print("- async_rouge_evaluator: 비동기 ROUGE 평가")
print("- async_length_evaluator: 비동기 응답 길이 평가")

비동기 함수 정의 완료
- async_rag_task: LangChain ainvoke()를 사용한 비동기 RAG Task
- async_rouge_evaluator: 비동기 ROUGE 평가
- async_length_evaluator: 비동기 응답 길이 평가


In [14]:
import time
import asyncio
from langfuse import get_client

langfuse_client = get_client()

# ============================================
# 0) 로컬 실험용 데이터 포맷 변환 (Langfuse 규격 맞춤)
# ============================================
# Langfuse 로컬 run_experiment는 각 데이터 아이템에 'input'과 'expected_output' 키가 필수입니다.
formatted_data = [
    {
        "input": x["user_input"],
        "expected_output": x["reference"],
    }
    for x in data
]

# ============================================
# 1) 동기(Sync) 실험용 Task 정의
# ============================================
def local_rag_task(*, item, **kwargs):
    # 포맷된 딕셔너리에서 'input' 추출
    question = item.get("input", "") 
    
    # 동기 체인 실행
    return normal_retriever.invoke(question)


# --- 동기 실험 실행 ---
print("▶️ 금융 데이터로 동기(Sync) 벤치마크 시작...")
start_sync = time.time()

sync_result = langfuse_client.run_experiment(
    name="Sync_Finance_RAG_Benchmark",
    description="금융 데이터셋 동기 실행 벤치마크",
    data=formatted_data,  # 👈 가공된 formatted_data 사용!
    task=local_rag_task,
    evaluators=[rouge_evaluator, length_evaluator],
    max_concurrency=1,   # 순차 실행
)

sync_elapsed = time.time() - start_sync


# ============================================
# 2) 비동기(Async) 실험용 Task 정의
# ============================================
async def async_local_rag_task(*, item, **kwargs):
    # 포맷된 딕셔너리에서 'input' 추출
    question = item.get("input", "") 
    
    # 비동기 체인 실행 (반드시 await와 ainvoke 사용)
    return await normal_retriever.ainvoke(question)


# --- 비동기 실험 실행 ---
print("▶️ 금융 데이터로 비동기(Async) 벤치마크 시작...")
start_async = time.time()

async_result = langfuse_client.run_experiment(
    name="Async_Finance_RAG_Benchmark",
    description="금융 데이터셋 비동기 실행 벤치마크",
    data=formatted_data,  # 👈 가공된 formatted_data 사용!
    task=async_local_rag_task,
    evaluators=[async_rouge_evaluator, async_length_evaluator],
    max_concurrency=5,   # 동시 5개 병렬 실행
)

async_elapsed = time.time() - start_async


# ============================================
# 3) 최종 결과 비교 출력
# ============================================
print("\n" + "=" * 60)
print(f" 📊 동기 실행 시간 (max_concurrency=1) : {sync_elapsed:.2f}초")
print(f" 📊 비동기 실행 시간 (max_concurrency=5): {async_elapsed:.2f}초")
speedup = sync_elapsed / async_elapsed if async_elapsed > 0 else 0
print(f" 🚀 비동기 전환 후 속도 향상       : {speedup:.1f}x 가속됨")
print("=" * 60 + "\n")

# 실험 결과 요약 출력
print(async_result.format())


▶️ 금융 데이터로 동기(Sync) 벤치마크 시작...
▶️ 금융 데이터로 비동기(Async) 벤치마크 시작...

 📊 동기 실행 시간 (max_concurrency=1) : 8.94초
 📊 비동기 실행 시간 (max_concurrency=5): 3.87초
 🚀 비동기 전환 후 속도 향상       : 2.3x 가속됨

Individual Results: Hidden (30 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: Async_Finance_RAG_Benchmark
📋 Run name: Async_Finance_RAG_Benchmark - 2026-07-04T16:23:45.816242Z - 금융 데이터셋 비동기 실행 벤치마크
30 items
Evaluations:
  • response_length
  • rouge_l

Average Scores:
  • response_length: 1314.367
  • rouge_l: 0.066



In [20]:
import pandas as pd

# 1. 랭퓨즈 클라이언트로부터 지정한 데이터셋 가져오기
dataset = langfuse_client.get_dataset(name="RAG_Evaluation_practice_new_1")

# 2. 데이터셋 아이템들을 판다스 데이터프레임 구조에 맞게 리스트로 파싱
qa_list = []
for item in dataset.items:
    # 💡 Langfuse의 item.input이 딕셔너리 형태일 때와 문자열일 때 모두 대응 가능하도록 처리합니다.
    if isinstance(item.input, dict):
        user_input = item.input.get("user_input", item.input.get("input", ""))
        reference_contexts = item.input.get("reference_contexts", "")
    else:
        user_input = item.input
        reference_contexts = ""
        
    qa_list.append({
        # 기존 평가 가이드라인 및 판다스 컬럼명 규격에 맞춥니다.
        "user_input": user_input,
        "reference": item.expected_output if item.expected_output else "정답 없음",
        "reference_contexts": reference_contexts,
        # ⭐ [추가] Langfuse 아이템 고유의 소스 트레이스 ID 추출 (없으면 None)
        "trace_id": getattr(item, "source_trace_id", "트레이스 없음") 
    })

# 3. 리스트를 데이터프레임으로 최종 변환
df_qa_test = pd.DataFrame(qa_list)

# ✏️ 샘플 수 제한 기능
# MAX_SAMPLES = 15
# df_qa_test = df_qa_test.head(MAX_SAMPLES).reset_index(drop=True)

print(f"✅ Langfuse 데이터셋 로드 완료: {df_qa_test.shape[0]}개 문항 사용")

# 상위 3개 데이터 출력 확인 (이제 trace_id 컬럼도 함께 보입니다!)
df_qa_test.head(3)

✅ Langfuse 데이터셋 로드 완료: 90개 문항 사용


,user_input,reference,reference_contexts,trace_id
0,녹색금융 실행계획의 세 가지 목표를 각각 달성하기 위해 어떠한 구체적인 계획이 마련...,"녹색금융 실행계획은 정책금융 지원 확대, 민간금융 활성화 및 시장인프라 정비를 통해...","[녹색금융 실행계획의 세 가지 목표는 정책금융 선도적 지원, 민간가금 유입 유도, ...",None
1,"기후위기 그린뉴딜 기본법과 녹색금융 특별법에서 주요하게 다룬 내용은 무엇이며, 이를...",기후위기 그린뉴딜 기본법과 녹색금융 특별법은 금융지원 및 금융상품 개발을 주요 내용...,[기후위기 그린뉴딜 기본법과 녹색금융 특별법에서 주요하게 다룬 내용은 법의 활성화를...,None
2,TCFD의 '전략' 핵심요소는 어떤 기후변화 시나리오를 고려하면서 잠재적 재무 리스...,TCFD의 '전략' 핵심요소는 1.5~2도 섭씨의 기후변화 시나리오를 고려하여 단기...,[TCFD의 '전략' 핵심요소는 기후변화와 관련된 리스크 및 기회 정보가 조직의 비...,None


In [16]:
try:
    from langfuse.langchain import CallbackHandler
    from langfuse import get_client

    langfuse_handler = CallbackHandler()
    langfuse_client = get_client()
    LANGFUSE_ENABLED = True
    print("Langfuse 설정 완료 ✓")
except Exception as e:
    langfuse_handler = None
    langfuse_client = None
    LANGFUSE_ENABLED = False
    print(f"Langfuse 비활성화 (키 없음 또는 연결 실패): {e}")

Langfuse 설정 완료 ✓


In [ ]:
from openevals.llm import create_llm_as_judge

# 공통 한국어 Correctness 평가 프롬프트
# choices=[0.0, 0.5, 1.0]으로 이산형 점수 사용 → Cohen's Kappa 계산에 적합
KOREAN_CORRECTNESS_PROMPT = """당신은 RAG 시스템의 답변 정확성을 평가하는 전문가입니다.

<질문>
{inputs}
</질문>

<생성된 답변>
{outputs}
</생성된 답변>

<참조 답변>
{reference_outputs}
</reference_outputs>

다음 기준에 따라 반드시 아래 점수 중 하나를 선택하세요:

- **1.0**: 생성된 답변이 참조 답변과 동일하거나 모든 핵심 내용을 포함하고 사실적으로 정확함
- **0.5**: 생성된 답변이 참조 답변의 일부 핵심 내용을 포함하지만 일부 누락되거나 부정확함  
- **0.0**: 생성된 답변이 참조 답변과 다르거나 사실적으로 부정확함

반드시 1.0, 0.5, 0.0 중 하나만 선택하세요.
"""

# Langfuse Prompt Registry 연동
try:
    if LANGFUSE_ENABLED and langfuse_client:
        try:
            # Prompt Registry에서 프롬프트 가져오기
            langfuse_prompt = langfuse_client.get_prompt("rag_evals/korean-correctness-judge")
            KOREAN_CORRECTNESS_PROMPT = langfuse_prompt.prompt
            print("Langfuse Prompt Registry에서 'korean-correctness-judge' 프롬프트를 성공적으로 로드했습니다. ✓")
        except Exception:
            # 등록된 프롬프트가 없으면 신규 등록
            langfuse_client.create_prompt(
                name="korean-correctness-judge",
                type="text",
                prompt=KOREAN_CORRECTNESS_PROMPT,
                labels=["production"]
            )
            print("Langfuse Prompt Registry에 'korean-correctness-judge' 프롬프트를 신규 등록 및 로드했습니다. ✓")
except Exception as e:
    print(f"Langfuse Prompt Registry 연동 실패 (기본 로컬 프롬프트 사용): {e}")

# 3개 Judge 모델 생성 (동일한 프롬프트 + 다른 모델)
judge_gpt_mini = create_llm_as_judge(
    prompt=KOREAN_CORRECTNESS_PROMPT,
    feedback_key="correctness",
    model="openai:gpt-4.1",
    choices=[0.0, 0.5, 1.0],      # 이산형 선택지
)

judge_gpt_nano = create_llm_as_judge(
    prompt=KOREAN_CORRECTNESS_PROMPT,
    feedback_key="correctness",
    model="openai:gpt-4.1-nano",
    choices=[0.0, 0.5, 1.0],
)

judge_groq = create_llm_as_judge(
    prompt=KOREAN_CORRECTNESS_PROMPT,
    feedback_key="correctness",
    model="groq:llama-3.3-70b-versatile",
    choices=[0.0, 0.5, 1.0],
)

# Judge 딕셔너리 (이름 → judge 함수)
judges = {
    "GPT-4.1": judge_gpt_mini,
    "GPT-4.1-nano": judge_gpt_nano,
    "Llama-3.3-70b (Groq)": judge_groq,
}

print("Judge 모델 초기화 완료")
for name in judges:
    print(f"  - {name}")

Langfuse Prompt Registry에 'korean-correctness-judge' 프롬프트를 신규 등록 및 로드했습니다. ✓
Judge 모델 초기화 완료
  - GPT-4.1
  - GPT-4.1-nano
  - Llama-3.3-70b (Groq)


In [26]:
# 1단계: RAG 체인으로 전체 테스트셋 답변 생성 (Langfuse Tracing 포함)
print("RAG 체인으로 답변 생성 중...")
test_records = []

for i, row in df_qa_test.iterrows():
    question = str(row.get("user_input", ""))
    reference = str(row.get("reference", ""))
    
    # RAG 체인 실행 (Langfuse 콜백 전달 )
    config = {"callbacks": [langfuse_handler]} if LANGFUSE_ENABLED and langfuse_handler else {}
    answer = normal_retriever.invoke(question, config=config)
    
    # 생성된 Trace ID 저장
    trace_id = langfuse_handler.last_trace_id if LANGFUSE_ENABLED and langfuse_handler else None
    
    test_records.append({
        "idx": i,
        "question": question,
        "answer": answer,
        "reference": reference,
        "trace_id": trace_id
    })
    
    # 진행 상황 출력 (5개마다)
    if (i + 1) % 5 == 0:
        print(f"  {i + 1}/{len(df_qa_test)}개 완료...")

print(f"\n답변 생성 완료: {len(test_records)}개")

# 결과 데이터프레임 생성
df_answers = pd.DataFrame(test_records)

RAG 체인으로 답변 생성 중...
  5/90개 완료...
  10/90개 완료...
  15/90개 완료...
  20/90개 완료...
  25/90개 완료...
  30/90개 완료...
  35/90개 완료...
  40/90개 완료...
  45/90개 완료...
  50/90개 완료...
  55/90개 완료...
  60/90개 완료...
  65/90개 완료...
  70/90개 완료...
  75/90개 완료...
  80/90개 완료...
  85/90개 완료...
  90/90개 완료...

답변 생성 완료: 90개


In [36]:
import re

target_qa_list = test_records[:5]
df_qa_test_limited = df_answers.head(5).copy()

print(f"🔄 전체 {len(test_records)}개 중 상위 5개 문항에 대해 디버깅 평가를 시작합니다...")

# 각 판사별로 5개의 점수가 정확히 쌓이도록 리스트 초기화
judge_scores = {name: [] for name in judges}

def clean_text(value):
    if not value:
        return ""
    if isinstance(value, list):
        return "\n".join([clean_text(v) for v in value])
    if hasattr(value, "page_content"):
        return str(value.page_content)
    if isinstance(value, dict):
        return value.get("page_content", str(value))
    return str(value)

for i, record in enumerate(target_qa_list):
    user_question = clean_text(record.get("user_input", record.get("question", "")))
    user_answer = clean_text(record.get("answer", record.get("output", record.get("response", ""))))
    user_reference = clean_text(record.get("reference", record.get("expected_output", "정답 없음")))
    
    if not user_answer.strip():
        print(f"  ❌ [경고] idx={i}번 데이터의 답변 본문이 비어있습니다!")

    trace_id = record.get("trace_id")
    if not trace_id or trace_id == "트레이스 없음":
        import uuid
        trace_id = f"eval-trace-{uuid.uuid4()}" 

    for judge_name, judge_fn in judges.items():
        # 기본값 세팅
        score = 0.0
        
        try:
            # 1. Judge 호출 (이 안에서 'score' KeyError가 발생하더라도 이제 아래 except로 안전하게 격리됩니다)
            result = judge_fn(
                inputs=user_question,
                outputs=user_answer,
                reference_outputs=user_reference,
            )
            
            # 2. 결과 파싱 안전 검사
            if isinstance(result, dict):
                score_val = result.get("score")
                if score_val is not None:
                    score = float(score_val if not isinstance(score_val, bool) else (1.0 if score_val else 0.0))
                else:
                    # score 키가 없을 때 텍스트에서 숫자 추출 시도
                    numbers = re.findall(r"\d+\.\d+|\d+", str(result))
                    if numbers: score = float(numbers[0])
            else:
                comment = str(result)
                numbers = re.findall(r"\d+\.\d+|\d+", comment)
                if numbers: score = float(numbers[0])
                
            if i == 0:
                print(f"  💡 [{judge_name} 첫 결과 성공!] Score: {score}")

            # 3. Langfuse Score API 업로드
            if LANGFUSE_ENABLED and langfuse_client and trace_id:
                try:
                    if hasattr(langfuse_client, "score") and hasattr(langfuse_client.score, "create"):
                        langfuse_client.score.create(
                            trace_id=trace_id,
                            name=f"correctness-{judge_name.lower().split()[0].replace('.', '-')}",
                            value=float(score)
                        )
                    elif hasattr(langfuse_client, "create_score"):
                        langfuse_client.create_score(
                            trace_id=trace_id,
                            name=f"correctness-{judge_name.lower().split()[0].replace('.', '-')}",
                            value=float(score)
                        )
                except Exception:
                    pass
                    
        except Exception as e:
            # 🌟 내부 요인으로 에러 발생 시 로그를 찍고 점수는 0.0으로 확정 처리
            print(f"  ❌ [{judge_name}, idx={i}] 내부 처리 에러 구출: {str(e)}")
            score = 0.0
            
        finally:
            # 🌟 [가장 중요] 무슨 일이 있어도 리스트에 값을 채워넣어 판다스 행 개수가 안 맞는 문제를 원천 차단합니다.
            judge_scores[judge_name].append(score)
            
    if (i + 1) % 2 == 0:
        print(f"  {i + 1}/{len(target_qa_list)}개 처리 진행 중...")

print(f"\n모든 Judge 평가 완료")

if LANGFUSE_ENABLED and langfuse_client:
    langfuse_client.flush()

# 이제 모든 판사 컬럼의 데이터 길이가 정확히 5개로 보장됩니다.
for judge_name, scores in judge_scores.items():
    df_qa_test_limited[judge_name] = scores

df_qa_test_limited

🔄 전체 90개 중 상위 5개 문항에 대해 디버깅 평가를 시작합니다...
  💡 [GPT-4.1 첫 결과 성공!] Score: 1.0
  💡 [GPT-4.1-nano 첫 결과 성공!] Score: 1.0
  💡 [Llama-3.3-70b (Groq) 첫 결과 성공!] Score: 0.5
  2/5개 처리 진행 중...
  4/5개 처리 진행 중...

모든 Judge 평가 완료


,user_input,reference,reference_contexts,GPT-4.1,GPT-4.1-nano,Llama-3.3-70b (Groq)
0,녹색금융 실행계획의 세 가지 목표를 각각 달성하기 위해 어떠한 구체적인 계획이 마련...,"녹색금융 실행계획은 정책금융 지원 확대, 민간금융 활성화 및 시장인프라 정비를 통해...","[녹색금융 실행계획의 세 가지 목표는 정책금융 선도적 지원, 민간가금 유입 유도, ...",1.0,1.0,0.5
1,"기후위기 그린뉴딜 기본법과 녹색금융 특별법에서 주요하게 다룬 내용은 무엇이며, 이를...",기후위기 그린뉴딜 기본법과 녹색금융 특별법은 금융지원 및 금융상품 개발을 주요 내용...,[기후위기 그린뉴딜 기본법과 녹색금융 특별법에서 주요하게 다룬 내용은 법의 활성화를...,1.0,0.5,0.5
2,TCFD의 '전략' 핵심요소는 어떤 기후변화 시나리오를 고려하면서 잠재적 재무 리스...,TCFD의 '전략' 핵심요소는 1.5~2도 섭씨의 기후변화 시나리오를 고려하여 단기...,[TCFD의 '전략' 핵심요소는 기후변화와 관련된 리스크 및 기회 정보가 조직의 비...,1.0,1.0,0.5
3,"PRI, PSI, PRB에서 각각 공통적으로 강조하는 활동은 무엇이며, 이들 원칙에...","PRI, PSI, PRB에서 공통적으로 강조하는 활동은 지속가능성을 촉진하는 것이며...","[PRB는 비즈니스 전략을 SDGs, 파리기후협정 및 관련 국가·지역 프레임워크와 ...",0.5,0.0,0.5
4,"PRI, PSI, PRB에서 각각 공통적으로 강조하는 활동은 무엇이며, 이들 원칙에...","PRI, PSI, PRB 모두 ESG 이슈의 통합, 리스크 관리, 이해관계자와의 협...","[PRI, PSI, PRB 모두에서 공통적으로 강조하는 활동은 ESG(환경, 사회,...",1.0,1.0,0.5


In [37]:
from sklearn.metrics import cohen_kappa_score
import numpy as np
import itertools

judge_names = list(judges.keys())

# 모든 Judge 쌍에 대해 Cohen's Kappa 계산
kappa_matrix = pd.DataFrame(
    np.ones((len(judge_names), len(judge_names))),  # 대각선 = 1.0 (자기 자신과 일치)
    index=judge_names,
    columns=judge_names,
)

kappa_results = {}

for judge_a, judge_b in itertools.combinations(judge_names, 2):
    scores_a = df_answers[judge_a].values
    scores_b = df_answers[judge_b].values
    
    try:
        # 이산형 범주로 변환 (0.0, 0.5, 1.0 → 문자열 레이블)
        labels_a = [str(s) for s in scores_a]
        labels_b = [str(s) for s in scores_b]
        
        kappa = cohen_kappa_score(labels_a, labels_b)
        if np.isnan(kappa):  # 두 Judge 점수가 완전히 동일한 경우 (0/0)
            kappa = 1.0
    except Exception as e:
        print(f"  계산 오류 ({judge_a} vs {judge_b}): {e}")
        kappa = float("nan")
    
    kappa_matrix.loc[judge_a, judge_b] = kappa
    kappa_matrix.loc[judge_b, judge_a] = kappa  # 대칭 행렬
    kappa_results[f"{judge_a} vs {judge_b}"] = kappa
    
    # Kappa 해석
    def interpret_kappa(k):
        if np.isnan(k): return "완전 동일 (1.0으로 처리)"
        if k < 0:    return "일치 없음"
        elif k < 0.20: return "매우 낮음"
        elif k < 0.40: return "낮음"
        elif k < 0.60: return "보통"
        elif k < 0.80: return "높음"
        else:          return "거의 완벽"
    
    print(f"  {judge_a} vs {judge_b}: κ = {kappa:.4f} ({interpret_kappa(kappa)})")

print(f"\nCohen's Kappa 행렬:")
print(kappa_matrix.round(4))

  GPT-4.1 vs GPT-4.1-nano: κ = 1.0000 (거의 완벽)
  GPT-4.1 vs Llama-3.3-70b (Groq): κ = 1.0000 (거의 완벽)
  GPT-4.1-nano vs Llama-3.3-70b (Groq): κ = 1.0000 (거의 완벽)

Cohen's Kappa 행렬:
                      GPT-4.1  GPT-4.1-nano  Llama-3.3-70b (Groq)
GPT-4.1                   1.0           1.0                   1.0
GPT-4.1-nano              1.0           1.0                   1.0
Llama-3.3-70b (Groq)      1.0           1.0                   1.0


c:\Users\jhs92\modullm7\etf-bot\.venv\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\jhs92\modullm7\etf-bot\.venv\Lib\site-packages\sklearn\utils\_param_validation.py:218: UndefinedMetricWarning: `y1`, `y2` and `labels` have only one label in common. `cohen_kappa_score` is undefined and set to the value defined by the the `replace_undefined_by` param, which is set to nan.
  return func(*args, **kwargs)
c:\Users\jhs92\modullm7\etf-bot\.venv\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\jhs92\modullm7\etf-bot\.venv\Lib\site-packages\sklearn\utils\_param_validation.py:218: UndefinedM

In [38]:
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

# 평가 행렬 구성 (각 행: 하나의 샘플, 각 열: judge 점수)
# aggregate_raters는 평가자들의 범주형 레이블 행렬을 받아 집계합니다
ratings_matrix = df_answers[judge_names].values  # shape: (n_samples, n_judges)

# statsmodels aggregate_raters: 점수를 범주 빈도 행렬로 변환
# 점수를 정수 범주로 매핑 (0.0→0, 0.5→1, 1.0→2)
score_map = {0.0: 0, 0.5: 1, 1.0: 2}
int_ratings = np.vectorize(lambda x: score_map.get(x, 0))(ratings_matrix)

# aggregate_raters로 범주 빈도 행렬 생성
table, categories = aggregate_raters(int_ratings)

# Fleiss' Kappa 계산
fleiss_k = fleiss_kappa(table)

print(f"Fleiss' Kappa (3개 Judge 전체): κ = {fleiss_k:.4f}")
print(f"해석: {interpret_kappa(fleiss_k)}")

# categories: aggregate_raters가 반환한 실제 등장 범주 목록
label_map = {0: "0.0점", 1: "0.5점", 2: "1.0점"}
col_names = [label_map.get(cat, str(cat)) for cat in categories]
print("\n=== 범주 빈도 테이블 (처음 5행) ===")
print(pd.DataFrame(table[:5], columns=col_names))

Fleiss' Kappa (3개 Judge 전체): κ = nan
해석: 완전 동일 (1.0으로 처리)

=== 범주 빈도 테이블 (처음 5행) ===
   0.0점
0     3
1     3
2     3
3     3
4     3


c:\Users\jhs92\modullm7\etf-bot\.venv\Lib\site-packages\statsmodels\stats\inter_rater.py:266: RuntimeWarning: invalid value encountered in scalar divide
  kappa = (p_mean - p_mean_exp) / (1- p_mean_exp)
